## **Connect Colab**

In [ ]:
import os, sys, subprocess
from getpass import getpass

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "real-video"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "KEY"

## **Download DATASET**

In [ ]:
from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
!pip install -q yt-dlp

In [ ]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

## **Setup (30 classes)**

We use 30 classes. 10 base classes, then two new tasks of 10 classes each. The base model already knows the first 10.

In [ ]:
cfg.TASK_1 = ["ApplyEyeMakeup", "ApplyLipstick", "Archery", "BabyCrawling", "BalanceBeam",
              "BandMarching", "BlowDryHair", "BlowingCandles", "BodyWeightSquats", "Bowling"]
cfg.TASK_2 = ["BoxingPunchingBag", "BoxingSpeedBag", "BrushingTeeth", "CliffDiving", "CricketBowling",
              "CricketShot", "CuttingInKitchen", "FieldHockeyPenalty", "Haircut", "SoccerPenalty"]

In [ ]:
ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2
class_to_idx     = {name: i for i, name in enumerate(ALL_CLASSES_LIST)}
n_base = len(cfg.SELECTED_CLASSES)   # 10 base
n_t1   = len(cfg.TASK_1)             # 10 new in task 1
n_t2   = len(cfg.TASK_2)             # 10 new in task 2
print(f"base {n_base}  task1 {n_t1}  task2 {n_t2}  total {len(ALL_CLASSES_LIST)}")

## **Preprocess data**

In [ ]:
base_split  = build_group_split(cfg.SELECTED_CLASSES, train_groups=18, val_groups=3)
task1_split = build_group_split(cfg.TASK_1,           train_groups=18, val_groups=3)
task2_split = build_group_split(cfg.TASK_2,           train_groups=18, val_groups=3)

In [ ]:
# turn videos into clips (one group of source videos never leaks across train and test)
preprocess_group_split(base_split,  cfg.SELECTED_CLASSES, output_root=cfg.BASE_ROOT,  max_samples=cfg.MAX_SAMPLES)
preprocess_group_split(task1_split, cfg.TASK_1,           output_root=cfg.TASK1_ROOT, max_samples=cfg.MAX_SAMPLES)
preprocess_group_split(task2_split, cfg.TASK_2,           output_root=cfg.TASK2_ROOT, max_samples=cfg.MAX_SAMPLES)

## **Create Loaders**

In [ ]:
base_train  = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "train"), class_to_idx=class_to_idx)
base_val    = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "val"),   class_to_idx=class_to_idx)
base_test   = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "test"),  class_to_idx=class_to_idx)
task1_train = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "train"), class_to_idx=class_to_idx)
task1_val   = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "val"),   class_to_idx=class_to_idx)
task1_test  = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "test"),  class_to_idx=class_to_idx)
task2_train = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "train"), class_to_idx=class_to_idx)
task2_val   = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "val"),   class_to_idx=class_to_idx)
task2_test  = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "test"),  class_to_idx=class_to_idx)

base_train_loader  = DataLoader(base_train,  batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
base_test_loader   = DataLoader(base_test,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
task1_test_loader  = DataLoader(task1_test,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
task2_test_loader  = DataLoader(task2_test,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
task1_train_loader = DataLoader(task1_train, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
task1_val_loader   = DataLoader(task1_val,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
task2_train_loader = DataLoader(task2_train, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
task2_val_loader   = DataLoader(task2_val,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

# test set for all classes seen so far (20 then 30)
combined_test_loader2 = DataLoader(ConcatDataset([base_test, task1_test, task2_test]), batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
print(f"base {len(base_train)}/{len(base_test)}  task1 {len(task1_train)}/{len(task1_test)}  task2 {len(task2_train)}/{len(task2_test)}")

## **Part 1: Upload the base model (ResNet50, 10 classes)**

We load the ResNet50 model that was already trained on the 10 base classes. This is the teacher for the next steps. We test it on the base classes to check it works.

In [ ]:
# load the base ResNet50 (already trained on the 10 base classes)
teacher = ResNet50LSTM(num_classes=n_base).to(device)
teacher.load_state_dict(torch.load(cfg.RESNET50_PATH, map_location=device))
teacher.eval()

In [ ]:
# test the base model on the base classes
base_acc, base_preds, base_labels = test_model(teacher, base_test_loader, device)
plot_confusion_matrix(base_labels, base_preds, num_classes=n_base,
                      classes_list=cfg.SELECTED_CLASSES, name="Base ResNet50 (10 classes)")

## **Part 2: Continual learning with replay**

Add the new classes step by step: 10 to 20 to 30. We keep a small buffer of old clips and mix them into every new batch. Replay was the best method in the continual learning study. We plot the same charts as the CL notebook: training curves, confusion matrix, old vs new accuracy, and the forgetting curve.

In [ ]:
# the continual model starts from the base teacher
model = unfreeze_all(teacher).to(device)
forget_hist = []

# buffer of old clips: 15 base clips per class (big enough for base + task 1 later)
buffer = ReplayBuffer(max_size=600)
fill_buffer(buffer, base_train_loader, per_class=15)
print(f"buffer: {len(buffer.data)} clips")

**Task 1 (grow to 20 classes)**

In [ ]:
# grow the head to 20 classes and learn task 1 with replay
model = expand_classifier(model, n_base + n_t1)
model = unfreeze_all(model).to(device)

set_seed(cfg.SEED)
track1 = {"Base": base_test_loader, "Task 1": task1_test_loader}
hist_t1 = train_continual(model, task1_train_loader, task1_val_loader, device,
                          buffer=buffer, num_old_classes=n_base,
                          epochs=cfg.CL_EPOCHS, lr=cfg.CL_LR,
                          track_loaders=track1, track_history=forget_hist)
torch.cuda.empty_cache()

In [ ]:
plot_training_results(hist_t1, task_name="Task 1 - Replay")

**Task 2 (grow to 30 classes)**

In [ ]:
# add task 1 clips to the buffer, then grow to 30 and learn task 2
fill_buffer(buffer, task1_train_loader, per_class=15)
model = expand_classifier(model, n_base + n_t1 + n_t2)
model = unfreeze_all(model).to(device)

set_seed(cfg.SEED)
track2 = {"Base": base_test_loader, "Task 1": task1_test_loader, "Task 2": task2_test_loader}
hist_t2 = train_continual(model, task2_train_loader, task2_val_loader, device,
                          buffer=buffer, num_old_classes=n_base + n_t1,
                          epochs=cfg.CL_EPOCHS, lr=cfg.CL_LR,
                          track_loaders=track2, track_history=forget_hist)
torch.cuda.empty_cache()

In [ ]:
plot_training_results(hist_t2, task_name="Task 2 - Replay")

In [ ]:
# test the final continual model on all 30 classes
cl_acc, cl_preds, cl_labels = test_model(model, combined_test_loader2, device)
plot_confusion_matrix(cl_labels, cl_preds, num_classes=len(ALL_CLASSES_LIST),
                      classes_list=ALL_CLASSES_LIST, name="Continual model (30 classes)")

In [ ]:
# old vs new accuracy, like the CL notebook
split_at = n_base + n_t1
print_detailed_metrics(cl_labels, cl_preds, num_classes=len(ALL_CLASSES_LIST),
                       classes_list=ALL_CLASSES_LIST,
                       split_old=(0, split_at), split_new=(split_at, len(ALL_CLASSES_LIST)))

In [ ]:
# forgetting curve: how each task's accuracy moves over the whole training
plot_forgetting_history(forget_hist, epochs_per_task=cfg.CL_EPOCHS)

## **Part 3: Knowledge distillation (cosine)**

Copy the big 30 class continual model into a small MobileNet student. We use cosine distillation, the best method in the distillation study. We plot the training curve and the confusion matrix, like the distillation notebook.

In [ ]:
# the student learns all 30 classes at once
train_all_loader = DataLoader(ConcatDataset([base_train, task1_train, task2_train]), batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
val_all_loader   = DataLoader(ConcatDataset([base_val,   task1_val,   task2_val]),   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
student = MobileNetV3SmallLSTMStudent(num_classes=len(ALL_CLASSES_LIST)).to(device)
set_seed(cfg.SEED)
student, hist_kd = train_student(student, model, train_all_loader, val_all_loader, device,
                                 mode="cosine", T=cfg.KD_T, ce_weight=cfg.KD_CE_WEIGHT,
                                 distill_weight=cfg.KD_DISTILL_WEIGHT,
                                 epochs=cfg.KD_EPOCHS, lr=cfg.KD_LR)
torch.cuda.empty_cache()

In [ ]:
plot_training_results(hist_kd, task_name="Student CE + Cosine")

In [ ]:
student_acc, student_preds, student_labels = test_model(student, combined_test_loader2, device)
plot_confusion_matrix(student_labels, student_preds, num_classes=len(ALL_CLASSES_LIST),
                      classes_list=ALL_CLASSES_LIST, name="Student CE + Cosine")

In [ ]:
# how much smaller is the student
teacher_params = sum(p.numel() for p in model.parameters())
student_params = sum(p.numel() for p in student.parameters())
print(f"continual model : {cl_acc:.4f}  ({teacher_params/1e6:.1f}M params)")
print(f"student (cosine): {student_acc:.4f}  ({student_params/1e6:.1f}M params, {teacher_params/student_params:.0f}x smaller)")

## **Part 4: Few-shot learning (MAML)**

Teach the small student to learn a new class from only a few clips. We meta-train it on some new sports, then re-fit the 30 class head. Same steps and same settings as the MAML study notebook. We clear the GPU first.

In [ ]:
torch.cuda.empty_cache(); gc.collect()

# same settings as the MAML study notebook
N_WAY = 5; K_SHOT = 3; K_QUERY = 3
META_EPOCHS = 8; META_EPISODES = 60
INNER_STEPS = 5; META_INNER_LR = 0.05; META_LR = 1e-4
META_TRAIN_LAST = 4
HEAD_RECAL_EPOCHS = 3; HEAD_RECAL_LR = 1e-3
POOL_PER_CLASS = 6

# novel sports for meta-training (not among the 30 known classes)
known     = set(ALL_CLASSES_LIST)
MAML_META = [c for c in cfg.SPORTS_META if c not in known][:15]
maml_c2i  = {c: i for i, c in enumerate(ALL_CLASSES_LIST + MAML_META)}

ULAL_META_ROOT = f"{cfg.DATA_ROOT}/grouped_ulal_meta"
meta_split = build_group_split(MAML_META, train_groups=18, val_groups=3)
preprocess_group_split(meta_split, MAML_META, output_root=ULAL_META_ROOT, max_samples=8, max_test_samples=6)
meta_train_ds = UCF101Clips(os.path.join(ULAL_META_ROOT, "train"), class_to_idx=maml_c2i)
pool_clips, pool_labels = build_pool([meta_train_ds], per_class=POOL_PER_CLASS)
meta_ids = sorted(set(pool_labels.tolist()))
print(f"meta classes {len(MAML_META)} | pool {tuple(pool_clips.shape)}")

In [ ]:
set_seed(cfg.SEED)
maml_student = copy.deepcopy(student)
maml_student.fc = nn.Linear(cfg.HIDDEN_SIZE, N_WAY).to(device)   # throwaway 5-way head for episodes

# meta-train only the last blocks (keep base features), then re-fit the 30-way head
maml_student, hist_maml = meta_train_maml(maml_student, pool_clips, pool_labels, meta_ids,
                                          N_WAY, K_SHOT, K_QUERY, META_EPOCHS, META_EPISODES,
                                          META_INNER_LR, INNER_STEPS, META_LR, device,
                                          freeze_backbone=True, train_last=META_TRAIN_LAST)
maml_student = recalibrate_head(maml_student, train_all_loader, len(ALL_CLASSES_LIST), device,
                                HEAD_RECAL_EPOCHS, HEAD_RECAL_LR)
del pool_clips, pool_labels; gc.collect(); torch.cuda.empty_cache()

plt.figure(figsize=(7, 4))
plt.plot(range(1, META_EPOCHS + 1), hist_maml["query_accs"], marker="o")
plt.xlabel("meta epoch"); plt.ylabel("meta query accuracy"); plt.ylim(0, 1.05)
plt.title("MAML meta-training on new sports")
plt.show()

In [ ]:
# test the MAML student on the 30 known classes, like the MAML notebook
maml_acc, maml_preds, maml_labels = test_model(maml_student, combined_test_loader2, device)
print(f"MAML student on 30 classes: {maml_acc:.4f}  (KD student was {student_acc:.4f})")
plot_confusion_matrix(maml_labels, maml_preds, num_classes=len(ALL_CLASSES_LIST),
                      classes_list=ALL_CLASSES_LIST, name="MAML student (30 classes)")
torch.cuda.empty_cache()

## **Part 5: Test on real YouTube videos**

Test on real short videos from YouTube. We test the continual teacher (ResNet50) and the small student (MobileNet) on the same clips, then compare them in a table. Use classes the models already know. Put your own short video links below. YouTube can block downloads on Colab, so a local file or an archive.org link is safer.

In [ ]:
# short YouTube videos for classes the models know (must be in the 30 classes)
YT_VIDEOS = {
    "PushUps": ["https://www.youtube.com/watch?v=WDIpL0pjun0"],
    "Bowling": ["https://www.youtube.com/shorts/R3XK7QtASAk"],
}
# keep only classes the models know, then download the clips
yt_known_videos = {c: urls for c, urls in YT_VIDEOS.items() if c in class_to_idx}
print(f"testing known classes: {list(yt_known_videos)}")
yt_clips, yt_labels, yt_videos = download_youtube_clips(yt_known_videos, class_to_idx, clips_per_video=20)
print(f"downloaded {len(yt_clips)} clips")
torch.cuda.empty_cache()

In [ ]:
# look at one frame from the first and last clip
UCF101Clips.show_frame(yt_clips, yt_labels, clip_idx=0, frame_idx=0)
UCF101Clips.show_frame(yt_clips, yt_labels, clip_idx=len(yt_clips) - 1, frame_idx=0)

In [ ]:
# test both models on the same real clips
yt_loader = DataLoader(TensorDataset(yt_clips, yt_labels), batch_size=cfg.BATCH_SIZE)
teacher_yt_acc, _, _ = test_model(model,   yt_loader, device)
student_yt_acc, _, _ = test_model(student, yt_loader, device)
print(f"teacher (ResNet50) on YouTube : {teacher_yt_acc:.3f}")
print(f"student (MobileNet) on YouTube: {student_yt_acc:.3f}")
torch.cuda.empty_cache()

In [ ]:
# comparison table: teacher vs student on the same real clips
yt_rows = [
    {"model": "CL teacher (ResNet50)",  "youtube_acc": round(teacher_yt_acc, 3),
     "ucf_acc": round(cl_acc, 3),      "params_M": round(teacher_params / 1e6, 1)},
    {"model": "KD student (MobileNet)", "youtube_acc": round(student_yt_acc, 3),
     "ucf_acc": round(student_acc, 3), "params_M": round(student_params / 1e6, 1)},
]
yt_df = pd.DataFrame(yt_rows)
print(yt_df.to_string(index=False))

## **Summary**

In [ ]:
print("ULAL pipeline results")
print(f"  base ResNet50 (10 classes)  : {base_acc:.4f}")
print(f"  replay CL (30 classes)      : {cl_acc:.4f}")
print(f"  MobileNet student (cosine)  : {student_acc:.4f}  ({teacher_params/student_params:.0f}x smaller)")
print(f"  MAML student (30 classes)   : {maml_acc:.4f}")
print(f"  YouTube teacher vs student  : {teacher_yt_acc:.3f} vs {student_yt_acc:.3f}")

In [ ]:
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
payload = {"base_acc": base_acc, "continual_acc": cl_acc, "student_acc": student_acc,
           "maml_student_acc": maml_acc, "youtube_teacher_acc": teacher_yt_acc,
           "youtube_student_acc": student_yt_acc,
           "teacher_params": teacher_params, "student_params": student_params,
           "maml_curve": hist_maml["query_accs"]}
with open(f"{cfg.RESULTS_DIR}/ulal_summary.json", "w") as f:
    json.dump(payload, f, indent=2)
print(f"saved -> {cfg.RESULTS_DIR}/ulal_summary.json")